In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load files
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Ensure numeric 'value' columns where applicable
for df in (metric_df, trace_df, log_df):
    if 'value' in df.columns:
        df['value'] = pd.to_numeric(df['value'], errors='coerce')

# Helper to compute time range strings in UTC+8
def time_range_strings(df, ts_col='timestamp'):
    if ts_col not in df.columns or df[ts_col].dropna().empty:
        return None, None
    mn = pd.to_datetime(int(df[ts_col].min()), unit='s', utc=True).tz_convert(tz)
    mx = pd.to_datetime(int(df[ts_col].max()), unit='s', utc=True).tz_convert(tz)
    return mn.isoformat(), mx.isoformat()

# PROCESS metric.csv
metric_min_ts, metric_max_ts = time_range_strings(metric_df, 'timestamp')
g_metric = metric_df.groupby(['cmdb_id', 'kpi_name'])['value']
metric_counts = g_metric.count().to_frame('row_count')
metric_p95 = g_metric.quantile(0.95).to_frame('p95_value')
metric_summary = metric_counts.join(metric_p95).reset_index()
metric_summary = metric_summary.rename(columns={'kpi_name': 'name_field'})
metric_summary['file'] = 'metric.csv'
metric_summary = metric_summary[['file', 'cmdb_id', 'name_field', 'row_count', 'p95_value']]
metric_summary = metric_summary.sort_values('row_count', ascending=False).head(50)
# attach overall time range columns (same for all rows for compact reporting)
metric_summary['min_timestamp'] = metric_min_ts
metric_summary['max_timestamp'] = metric_max_ts

# PROCESS trace.csv
trace_min_ts, trace_max_ts = time_range_strings(trace_df, 'timestamp')
g_trace = trace_df.groupby(['cmdb_id', 'trace_name'])['value']
trace_counts = g_trace.count().to_frame('row_count')
trace_p95 = g_trace.quantile(0.95).to_frame('p95_value')
trace_summary = trace_counts.join(trace_p95).reset_index()
trace_summary = trace_summary.rename(columns={'trace_name': 'name_field'})
trace_summary['file'] = 'trace.csv'
trace_summary = trace_summary[['file', 'cmdb_id', 'name_field', 'row_count', 'p95_value']]
trace_summary = trace_summary.sort_values('row_count', ascending=False).head(50)
trace_summary['min_timestamp'] = trace_min_ts
trace_summary['max_timestamp'] = trace_max_ts

# PROCESS log.csv
log_min_ts, log_max_ts = time_range_strings(log_df, 'timestamp')
g_log = log_df.groupby(['cmdb_id', 'log_name'])['value']
log_counts = g_log.count().to_frame('row_count')
log_p95 = g_log.quantile(0.95).to_frame('p95_value')
log_summary = log_counts.join(log_p95).reset_index()
log_summary = log_summary.rename(columns={'log_name': 'name_field'})
log_summary['file'] = 'log.csv'
log_summary = log_summary[['file', 'cmdb_id', 'name_field', 'row_count', 'p95_value']]
log_summary = log_summary.sort_values('row_count', ascending=False).head(50)
log_summary['min_timestamp'] = log_min_ts
log_summary['max_timestamp'] = log_max_ts

# PROCESS error_logs.csv: compact single-row summary
error_total_rows = len(error_df)
distinct_cmdb = error_df['cmdb_id'].dropna().unique()[:10]  # up to 10 distinct cmdb_id
distinct_cmdb_str = ','.join(map(str, distinct_cmdb)) if len(distinct_cmdb) > 0 else ''
# earliest 5 timestamps (converted to UTC+8 ISO)
if 'timestamp' in error_df.columns and not error_df['timestamp'].dropna().empty:
    earliest_ts_vals = error_df['timestamp'].dropna().astype(int).sort_values().unique()[:5]
    earliest_ts_iso = [pd.to_datetime(int(t), unit='s', utc=True).tz_convert(tz).isoformat() for t in earliest_ts_vals]
    earliest_ts_str = ','.join(earliest_ts_iso)
else:
    earliest_ts_str = ''

error_summary = pd.DataFrame([{
    'file': 'error_logs.csv',
    'total_rows': error_total_rows,
    'sample_cmdb_ids_up_to_10': distinct_cmdb_str,
    'earliest_5_timestamps_UTC+8': earliest_ts_str
}])

# Display compact tables (one per file)
metric_summary, trace_summary, log_summary, error_summary

```
Out[1]:
```
```python
# Summarize the previously computed compact tables into a plain-English string.
# Reuse existing variables: metric_summary, trace_summary, log_summary, error_summary

def format_top_rows(df, n=3):
    if df is None or df.empty:
        return "  (no data)\n"
    rows = []
    top = df.sort_values('row_count', ascending=False).head(n)
    for _, r in top.iterrows():
        p95 = r.get('p95_value')
        # format numeric p95 compactly
        try:
            p95_s = f"{float(p95):.4g}"
        except Exception:
            p95_s = str(p95)
        rows.append(f"  - {r['cmdb_id']} | {r['name_field']} | rows={int(r['row_count'])} | p95={p95_s}")
    return "\n".join(rows) + "\n"

parts = []

# metric.csv summary
if 'metric_summary' in globals():
    ms = metric_summary
    parts.append("metric.csv:")
    parts.append(f" - Listed pairs returned: {ms.shape[0]} (top pairs by row count, limited to 50).")
    # time range (same for all rows)
    min_ts = ms['min_timestamp'].iloc[0] if 'min_timestamp' in ms.columns and not ms.empty else "N/A"
    max_ts = ms['max_timestamp'].iloc[0] if 'max_timestamp' in ms.columns and not ms.empty else "N/A"
    parts.append(f" - Timestamp range (UTC+8): {min_ts} to {max_ts}.")
    parts.append(" - Top pairs (up to 3):")
    parts.append(format_top_rows(ms, 3))
else:
    parts.append("metric.csv: (no summary available)")

# trace.csv summary
if 'trace_summary' in globals():
    ts = trace_summary
    parts.append("trace.csv:")
    parts.append(f" - Listed pairs returned: {ts.shape[0]} (top pairs by row count, limited to 50).")
    min_ts = ts['min_timestamp'].iloc[0] if 'min_timestamp' in ts.columns and not ts.empty else "N/A"
    max_ts = ts['max_timestamp'].iloc[0] if 'max_timestamp' in ts.columns and not ts.empty else "N/A"
    parts.append(f" - Timestamp range (UTC+8): {min_ts} to {max_ts}.")
    parts.append(" - Top pairs (up to 3):")
    parts.append(format_top_rows(ts, 3))
else:
    parts.append("trace.csv: (no summary available)")

# log.csv summary
if 'log_summary' in globals():
    ls = log_summary
    parts.append("log.csv:")
    parts.append(f" - Listed pairs returned: {ls.shape[0]} (top pairs by row count, limited to 50).")
    min_ts = ls['min_timestamp'].iloc[0] if 'min_timestamp' in ls.columns and not ls.empty else "N/A"
    max_ts = ls['max_timestamp'].iloc[0] if 'max_timestamp' in ls.columns and not ls.empty else "N/A"
    parts.append(f" - Timestamp range (UTC+8): {min_ts} to {max_ts}.")
    parts.append(" - Top pairs (up to 3):")
    parts.append(format_top_rows(ls, 3))
else:
    parts.append("log.csv: (no summary available)")

# error_logs.csv summary
if 'error_summary' in globals() and not error_summary.empty:
    er = error_summary.iloc[0]
    parts.append("error_logs.csv:")
    parts.append(f" - Total rows: {int(er['total_rows'])}")
    sample_cmdbs = er.get('sample_cmdb_ids_up_to_10') or ''
    parts.append(f" - Up to 10 sample cmdb_id values: {sample_cmdbs}")
    earliest = er.get('earliest_5_timestamps_UTC+8') or ''
    parts.append(f" - Earliest up-to-5 timestamps (UTC+8): {earliest}")
else:
    parts.append("error_logs.csv: (no summary available)")

summary = "\n".join(parts)

# Display the compact plain-English summary string
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(            file   cmdb_id                                         name_field  row_count     p95_value              min_timestamp              max_timestamp
1590  metric.csv  apache01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount         30  4.000000e+00  2021-03-09T23:30:00+08:00  2021-03-09T23:59:00+08:00
6     metric.csv      IG01           JVM-Threads_7778_JVM_ThreadCount_Threads         30  7.500000e+01  2021-03-09T23:30:00+08:00  2021-03-09T23:59:00+08:00
3     metric.csv      IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed         30  5.795390e+07  2021-03-09T23:30:00+08:00  2021-03-09T23:59:00+08:00
2     metric.csv      IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed         30  7.763210e+08  2021-03-09T23:30:00+08:00  2021-03-09T23:59:00+08:00
0     metric.csv      IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax         30  2.040136e+09  2021-03-09T23:30:00+08:00  2021-03-09T23:59:00+08:00
...          ...       ...                                                ...        ...           ...                        ...                        ...
1398  metric.csv  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...         30  0.000000e+00  2021-03-09T23:30:00+08:00  2021-03-09T23:59:00+08:00
1399  metric.csv  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...         30  0.000000e+00  2021-03-09T23:30:00+08:00  2021-03-09T23:59:00+08:00
1388  metric.csv  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...         30  3.020000e+02  2021-03-09T23:30:00+08:00  2021-03-09T23:59:00+08:00
1400  metric.csv  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...         30  0.000000e+00  2021-03-09T23:30:00+08:00  2021-03-09T23:59:00+08:00
1401  metric.csv  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...         30  0.000000e+00  2021-03-09T23:30:00+08:00  2021-03-09T23:59:00+08:00

[50 rows x 7 columns],           file   cmdb_id                       name_field  row_count   p95_value              min_timestamp              max_timestamp
0    trace.csv      IG01         trace.self.duration_mean         30   52.918652  2021-03-09T23:30:00+08:00  2021-03-09T23:59:00+08:00
1    trace.csv      IG01          trace.self.duration_p95         30   60.000977  2021-03-09T23:30:00+08:00  2021-03-09T23:59:00+08:00
2    trace.csv      IG01             trace.self.row_count         30  413.600000  2021-03-09T23:30:00+08:00  2021-03-09T23:59:00+08:00
3    trace.csv      IG01  trace.to_Tomcat01.duration_mean         30  147.550826  2021-03-09T23:30:00+08:00  2021-03-09T23:59:00+08:00
4    trace.csv      IG01   trace.to_Tomcat01.duration_p95         30  195.775440  2021-03-09T23:30:00+08:00  2021-03-09T23:59:00+08:00
..         ...       ...                              ...        ...         ...                        ...                        ...
96   trace.csv      MG02         trace.self.duration_mean         30    9.733698  2021-03-09T23:30:00+08:00  2021-03-09T23:59:00+08:00
97   trace.csv      MG02          trace.self.duration_p95         30   60.001000  2021-03-09T23:30:00+08:00  2021-03-09T23:59:00+08:00
98   trace.csv      MG02             trace.self.row_count         30  331.650000  2021-03-09T23:30:00+08:00  2021-03-09T23:59:00+08:00
108  trace.csv      MG02  trace.to_dockerB2.duration_mean         30    0.148789  2021-03-09T23:30:00+08:00  2021-03-09T23:59:00+08:00
182  trace.csv  dockerA1             trace.self.row_count         30  557.500000  2021-03-09T23:30:00+08:00  2021-03-09T23:59:00+08:00

[50 rows x 7 columns],        file   cmdb_id        name_field  row_count  p95_value              min_timestamp              max_timestamp
12  log.csv  apache01   log.error_count         26     0.7500  2021-03-09T23:30:00+08:00  2021-03-09T23:55:00+08:00
13  log.csv  apache01     log.row_count         26   169.0000  2021-03-09T23:30:00+08:00  2021-03-09T23:55:00+08:00
14  log.csv  apache02   log.error_count         26     1.0000  2021-03-09T23:30:00+08:00  2021-03-09T23:55:00+08:00
15  log.csv  apache02     log.row_count         26   176.2500  2021-03-09T23:30:00+08:00  2021-03-09T23:55:00+08:00
6   log.csv  Tomcat03   log.error_count         25     0.0000  2021-03-09T23:30:00+08:00  2021-03-09T23:55:00+08:00
8   log.csv  Tomcat03     log.row_count         25    87.0000  2021-03-09T23:30:00+08:00  2021-03-09T23:55:00+08:00
11  log.csv  Tomcat04     log.row_count         25    91.0000  2021-03-09T23:30:00+08:00  2021-03-09T23:55:00+08:00
9   log.csv  Tomcat04   log.error_count         25     0.0000  2021-03-09T23:30:00+08:00  2021-03-09T23:55:00+08:00
0   log.csv  Tomcat01   log.error_count         24     0.0000  2021-03-09T23:30:00+08:00  2021-03-09T23:55:00+08:00
2   log.csv  Tomcat01     log.row_count         24    93.9500  2021-03-09T23:30:00+08:00  2021-03-09T23:55:00+08:00
3   log.csv  Tomcat02   log.error_count          6     0.0000  2021-03-09T23:30:00+08:00  2021-03-09T23:55:00+08:00
5   log.csv  Tomcat02     log.row_count          6     1.0000  2021-03-09T23:30:00+08:00  2021-03-09T23:55:00+08:00
4   log.csv  Tomcat02  log.gc_pause_max          6     0.0800  2021-03-09T23:30:00+08:00  2021-03-09T23:55:00+08:00
10  log.csv  Tomcat04  log.gc_pause_max          4     0.0985  2021-03-09T23:30:00+08:00  2021-03-09T23:55:00+08:00
7   log.csv  Tomcat03  log.gc_pause_max          3     0.0770  2021-03-09T23:30:00+08:00  2021-03-09T23:55:00+08:00
1   log.csv  Tomcat01  log.gc_pause_max          3     0.0970  2021-03-09T23:30:00+08:00  2021-03-09T23:55:00+08:00,              file  total_rows sample_cmdb_ids_up_to_10                        earliest_5_timestamps_UTC+8
0  error_logs.csv           6        apache02,apache01  2021-03-09T23:30:55+08:00,2021-03-09T23:31:04+...)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

# Reuse previously loaded metric_df and tz if available; otherwise load/recreate.
import pandas as pd
import pytz

try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure numeric values
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full metric_df (before any window filtering)
global_p95 = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).reset_index()
global_p95 = global_p95.rename(columns={'value': 'global_p95'})

# Merge global_p95 back to metric_df for later comparisons
metric_df = metric_df.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# 2) Convert timestamps to timezone-aware datetimes (UTC -> convert to Asia/Shanghai UTC+8)
metric_df['timestamp_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Define failure window (inclusive start, exclusive end)
window_start = pd.to_datetime('2021-03-09T23:30:00').tz_localize(tz)
window_end   = pd.to_datetime('2021-03-10T00:00:00').tz_localize(tz)

# 2) Filter metric_df to the failure window
metric_window = metric_df[(metric_df['timestamp_dt'] >= window_start) & (metric_df['timestamp_dt'] < window_end)].copy()

# 3) Detect anomaly points in the window where value >= that component-KPI's global_p95
# Ensure we exclude rows where global_p95 is NaN
metric_window = metric_window[~metric_window['global_p95'].isna()].copy()
metric_anomalies_window = metric_window[metric_window['value'] >= metric_window['global_p95']].copy()

# 4) Produce aggregate table
# window_row_count per pair (in-window total rows)
window_counts = metric_window.groupby(['cmdb_id', 'kpi_name']).size().rename('window_row_count').reset_index()

# anomaly counts per pair
anomaly_counts = metric_anomalies_window.groupby(['cmdb_id', 'kpi_name']).size().rename('anomaly_count_in_window').reset_index()

# earliest anomaly timestamp per pair (UTC+8 ISO)
earliest_anom_ts = metric_anomalies_window.groupby(['cmdb_id', 'kpi_name'])['timestamp_dt'].min().reset_index()
earliest_anom_ts['earliest_anomaly_timestamp_UTC+8'] = earliest_anom_ts['timestamp_dt'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
earliest_anom_ts = earliest_anom_ts.drop(columns=['timestamp_dt'])

# max value in window per pair
max_in_window = metric_window.groupby(['cmdb_id', 'kpi_name'])['value'].max().rename('max_value_in_window').reset_index()

# global_p95 dataframe is already available; rename column if needed
gp = global_p95.copy()

# Combine aggregates: start from anomaly_counts to include only pairs with anomalies
agg = anomaly_counts.merge(gp, on=['cmdb_id', 'kpi_name'], how='left') \
                     .merge(earliest_anom_ts, on=['cmdb_id', 'kpi_name'], how='left') \
                     .merge(max_in_window, on=['cmdb_id', 'kpi_name'], how='left') \
                     .merge(window_counts, on=['cmdb_id', 'kpi_name'], how='left')

# Safety: fill NaNs (shouldn't be necessary for anomaly pairs) and format columns
agg['global_p95'] = agg['global_p95'].astype(float)
agg['anomaly_count_in_window'] = agg['anomaly_count_in_window'].astype(int)
agg['window_row_count'] = agg['window_row_count'].fillna(0).astype(int)
agg['max_value_in_window'] = agg['max_value_in_window'].astype(float)

# Sort by anomaly_count desc, then max_value_in_window desc
agg = agg.sort_values(by=['anomaly_count_in_window', 'max_value_in_window'], ascending=[False, False])

# Limit to top 20 rows
agg_top20 = agg[['cmdb_id', 'kpi_name', 'global_p95', 'anomaly_count_in_window',
                 'earliest_anomaly_timestamp_UTC+8', 'max_value_in_window', 'window_row_count']].head(20)

# 5) Save detailed anomaly points into variable metric_anomalies_window (already created)
# Do not print the full anomalies; only return the aggregate table (agg_top20)
agg_top20

```
Out[2]:
```
```python
# Summarize the anomaly aggregate (agg_top20) into a plain-English string.
# Reuse existing variables from the previous steps: agg_top20

try:
    df = agg_top20.copy()
except NameError:
    # fallback: try to load from earlier variable names if different
    df = globals().get('agg_top20', None)
    if df is None:
        raise

# Basic counts
num_pairs = int(df.shape[0])
total_anomaly_points = int(df['anomaly_count_in_window'].sum()) if num_pairs > 0 else 0
earliest_anom = df['earliest_anomaly_timestamp_UTC+8'].min() if num_pairs > 0 else "N/A"
latest_anom = df['earliest_anomaly_timestamp_UTC+8'].max() if num_pairs > 0 else "N/A"

# Top 5 examples (or fewer)
top_n = min(5, num_pairs)
examples = []
for _, r in df.head(top_n).iterrows():
    examples.append(
        f"{r['cmdb_id']} | {r['kpi_name']} | anomalies={int(r['anomaly_count_in_window'])} | "
        f"global_p95={float(r['global_p95']):.4g} | max_in_window={float(r['max_value_in_window']):.4g}"
    )

summary_lines = [
    f"Anomaly summary for the specified failure window (2021-03-09T23:30:00+08:00 to 2021-03-10T00:00:00+08:00):",
    f"- Number of (cmdb_id, kpi_name) pairs with breaches (top results returned): {num_pairs}",
    f"- Total anomaly points (sum of breaches across these pairs in window): {total_anomaly_points}",
    f"- Earliest reported anomaly timestamp (UTC+8) among these pairs: {earliest_anom}",
    f"- Example top {top_n} pairs (cmdb_id | kpi_name | anomaly_count | global_p95 | max_in_window):"
]
summary_lines += [f"  {e}" for e in examples]

summary = "\n".join(summary_lines)

# Display the plain-English summary string
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name    global_p95  anomaly_count_in_window earliest_anomaly_timestamp_UTC+8  max_value_in_window  window_row_count
0         IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09                       30         2021-03-09T23:30:00+0800         2.040136e+09                30
90        IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09                       30         2021-03-09T23:30:00+0800         2.040136e+09                30
180       MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09                       30         2021-03-09T23:30:00+0800         2.040136e+09                30
272       MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09                       30         2021-03-09T23:30:00+0800         2.040136e+09                30
275       MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  9.471166e+08                       30         2021-03-09T23:30:00+0800         9.471166e+08                30
183       MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  5.069049e+08                       30         2021-03-09T23:30:00+0800         5.069049e+08                30
3         IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.795390e+07                       30         2021-03-09T23:30:00+0800         5.795390e+07                30
1172  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  4.053960e+05                       30         2021-03-09T23:30:00+0800         4.053960e+05                30
1389  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  3.303510e+05                       30         2021-03-09T23:30:00+0800         3.303510e+05                30
1280  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  3.300540e+05                       30         2021-03-09T23:30:00+0800         3.300540e+05                30
1498  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  3.248520e+05                       30         2021-03-09T23:30:00+0800         3.248520e+05                30
1186  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02                       30         2021-03-09T23:30:00+0800         5.000000e+02                30
1297  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02                       30         2021-03-09T23:30:00+0800         5.000000e+02                30
1406  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02                       30         2021-03-09T23:30:00+0800         5.000000e+02                30
1515  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02                       30         2021-03-09T23:30:00+0800         5.000000e+02                30
1184  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  1.000000e+02                       30         2021-03-09T23:30:00+0800         1.000000e+02                30
1295  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  1.000000e+02                       30         2021-03-09T23:30:00+0800         1.000000e+02                30
1404  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  1.000000e+02                       30         2021-03-09T23:30:00+0800         1.000000e+02                30
1513  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  1.000000e+02                       30         2021-03-09T23:30:00+0800         1.000000e+02                30
806    Mysql02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  2.400000e+01                       30         2021-03-09T23:30:00+0800         2.400000e+01                30```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# Reuse trace_df and tz if available, otherwise load/define
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure numeric 'value'
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# 1) Compute global P95 per (cmdb_id, trace_name) using the full trace_df (before any window filtering)
trace_global_p95 = trace_df.groupby(['cmdb_id', 'trace_name'])['value'].quantile(0.95).reset_index()
trace_global_p95 = trace_global_p95.rename(columns={'value': 'global_p95'})

# Merge global_p95 into trace_df for later comparisons
trace_df = trace_df.merge(trace_global_p95, on=['cmdb_id', 'trace_name'], how='left')

# 2) Convert timestamps to timezone-aware datetimes (input timestamps are seconds UTC; convert to UTC+8)
trace_df['timestamp_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Define failure window (inclusive start, exclusive end)
window_start = pd.to_datetime('2021-03-09T23:30:00').tz_localize(tz)
window_end   = pd.to_datetime('2021-03-10T00:00:00').tz_localize(tz)

# 2) Filter trace_df to the failure window
trace_window = trace_df[(trace_df['timestamp_dt'] >= window_start) & (trace_df['timestamp_dt'] < window_end)].copy()

# 3) Detect anomaly points where value >= global_p95 (exclude rows with missing global_p95)
trace_window = trace_window[~trace_window['global_p95'].isna()].copy()
trace_anomalies_window = trace_window[trace_window['value'] >= trace_window['global_p95']].copy()

# 4) Produce aggregate table
# window_row_count per pair
window_counts = trace_window.groupby(['cmdb_id', 'trace_name']).size().rename('window_row_count').reset_index()

# anomaly counts per pair
anomaly_counts = trace_anomalies_window.groupby(['cmdb_id', 'trace_name']).size().rename('anomaly_count_in_window').reset_index()

# earliest anomaly timestamp per pair (UTC+8 ISO)
earliest_anom_ts = trace_anomalies_window.groupby(['cmdb_id', 'trace_name'])['timestamp_dt'].min().reset_index()
earliest_anom_ts['earliest_anomaly_timestamp_UTC+8'] = earliest_anom_ts['timestamp_dt'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
earliest_anom_ts = earliest_anom_ts.drop(columns=['timestamp_dt'])

# max value in window per pair
max_in_window = trace_window.groupby(['cmdb_id', 'trace_name'])['value'].max().rename('max_value_in_window').reset_index()

# Combine aggregates, starting from anomaly_counts to include only pairs with anomalies
agg = anomaly_counts.merge(trace_global_p95, on=['cmdb_id', 'trace_name'], how='left') \
                    .merge(earliest_anom_ts, on=['cmdb_id', 'trace_name'], how='left') \
                    .merge(max_in_window, on=['cmdb_id', 'trace_name'], how='left') \
                    .merge(window_counts, on=['cmdb_id', 'trace_name'], how='left')

# Ensure types and sort
agg['global_p95'] = agg['global_p95'].astype(float)
agg['anomaly_count_in_window'] = agg['anomaly_count_in_window'].astype(int)
agg['max_value_in_window'] = agg['max_value_in_window'].astype(float)
agg['window_row_count'] = agg['window_row_count'].fillna(0).astype(int)

# Sort and limit to top 20 rows
agg = agg.sort_values(by=['anomaly_count_in_window', 'max_value_in_window'], ascending=[False, False])
trace_agg_top20 = agg[['cmdb_id', 'trace_name', 'global_p95', 'anomaly_count_in_window',
                       'earliest_anomaly_timestamp_UTC+8', 'max_value_in_window', 'window_row_count']].head(20)

# 5) trace_anomalies_window is saved for later use (do not display full rows)
# Return only the compact aggregate table
trace_agg_top20

```
Out[3]:
```
```python
# Summarize the trace anomaly aggregate (trace_agg_top20) into a plain-English string.
# This is a summarization task so we return the summary string directly.
try:
    df = trace_agg_top20.copy()
except NameError:
    df = globals().get('trace_agg_top20', None)
    if df is None:
        raise RuntimeError("trace_agg_top20 not found in the environment. Run the previous analysis steps first.")

num_pairs = int(df.shape[0])
total_anomaly_points = int(df['anomaly_count_in_window'].sum()) if num_pairs > 0 else 0
earliest_anom = df['earliest_anomaly_timestamp_UTC+8'].min() if num_pairs > 0 else "N/A"
latest_anom = df['earliest_anomaly_timestamp_UTC+8'].max() if num_pairs > 0 else "N/A"

# Prepare top example lines (up to 5)
examples = []
top_n = min(5, num_pairs)
for _, r in df.head(top_n).iterrows():
    try:
        gp = f"{float(r['global_p95']):.6g}"
    except Exception:
        gp = str(r['global_p95'])
    try:
        mv = f"{float(r['max_value_in_window']):.6g}"
    except Exception:
        mv = str(r['max_value_in_window'])
    examples.append(f"{r['cmdb_id']} | {r['trace_name']} | anomalies={int(r['anomaly_count_in_window'])} | global_p95={gp} | max_in_window={mv}")

summary_lines = [
    "Trace anomalies summary for window 2021-03-09T23:30:00+08 to 2021-03-10T00:00:00+08 (using global P95 per (cmdb_id,trace_name)):",
    f"- Number of (cmdb_id, trace_name) pairs with breaches (top results shown): {num_pairs}",
    f"- Total anomaly points across these pairs in the window: {total_anomaly_points}",
    f"- Earliest anomaly timestamp (UTC+8) among these pairs: {earliest_anom}",
    f"- Latest anomaly timestamp (UTC+8) among these pairs: {latest_anom}",
    f"- Example top {top_n} pairs (cmdb_id | trace_name | anomaly_count | global_p95 | max_in_window):"
]
summary_lines += [f"  {line}" for line in examples]

summary = "\n".join(summary_lines)

# Display the summary string (as the final variable)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                      trace_name  global_p95  anomaly_count_in_window earliest_anomaly_timestamp_UTC+8  max_value_in_window  window_row_count
55       MG01         trace.self.duration_p95   60.001000                        4         2021-03-09T23:30:00+0800              60.0010                30
97       MG02         trace.self.duration_p95   60.001000                        3         2021-03-09T23:30:00+0800              60.0010                30
86       MG02   trace.from_dockerA2.row_count   13.000000                        3         2021-03-09T23:38:00+0800              14.0000                27
203  dockerA2         trace.to_MG02.row_count   13.000000                        3         2021-03-09T23:38:00+0800              14.0000                27
212  dockerB1            trace.self.row_count  909.950000                        2         2021-03-09T23:42:00+0800            1202.0000                30
230  dockerB2            trace.self.row_count  878.350000                        2         2021-03-09T23:41:00+0800            1137.0000                30
137  Tomcat02            trace.self.row_count  767.550000                        2         2021-03-09T23:58:00+0800             963.0000                30
152  Tomcat03            trace.self.row_count  724.700000                        2         2021-03-09T23:58:00+0800             954.0000                30
167  Tomcat04            trace.self.row_count  678.050000                        2         2021-03-09T23:57:00+0800             894.0000                30
122  Tomcat01            trace.self.row_count  730.050000                        2         2021-03-09T23:58:00+0800             891.0000                30
182  dockerA1            trace.self.row_count  557.500000                        2         2021-03-09T23:52:00+0800             696.0000                30
197  dockerA2            trace.self.row_count  530.600000                        2         2021-03-09T23:48:00+0800             690.0000                30
17       IG02            trace.self.row_count  414.100000                        2         2021-03-09T23:58:00+0800             596.0000                30
2        IG01            trace.self.row_count  413.600000                        2         2021-03-09T23:58:00+0800             540.0000                30
56       MG01            trace.self.row_count  320.150000                        2         2021-03-09T23:42:00+0800             385.0000                30
98       MG02            trace.self.row_count  331.650000                        2         2021-03-09T23:42:00+0800             371.0000                30
7        IG01  trace.to_Tomcat02.duration_p95  211.653697                        2         2021-03-09T23:30:00+0800             274.5086                30
130  Tomcat02    trace.from_IG01.duration_p95  211.653697                        2         2021-03-09T23:30:00+0800             274.5086                30
22       IG02  trace.to_Tomcat02.duration_p95  213.196722                        2         2021-03-09T23:30:00+0800             272.0222                30
133  Tomcat02    trace.from_IG02.duration_p95  213.196722                        2         2021-03-09T23:30:00+0800             272.0222                30```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing dataframes if present, otherwise load
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')

try:
    error_df
except NameError:
    error_df = pd.read_csv('error_logs.csv')

# Ensure numeric 'value' in log_df
if 'value' in log_df.columns:
    log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# Timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Create timezone-aware datetime columns (timestamps are in seconds UTC)
log_df['timestamp_dt'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
error_df['timestamp_dt'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Define failure window (inclusive start, exclusive end)
window_start = pd.to_datetime('2021-03-09T23:30:00').tz_localize(tz)
window_end   = pd.to_datetime('2021-03-10T00:00:00').tz_localize(tz)

# Candidate components
candidates = ['IG01','IG02','Tomcat01','Tomcat02','Tomcat03','Tomcat04','MG01','MG02','apache01','apache02']

# 1) Filter log.csv rows for candidates within window and save for later
candidate_logs_window = log_df[
    (log_df['cmdb_id'].isin(candidates)) &
    (log_df['timestamp_dt'] >= window_start) &
    (log_df['timestamp_dt'] < window_end)
].copy()

# Prepare helper column for counting log.error_count > 0 occurrences
candidate_logs_window['error_nonzero'] = np.where(
    (candidate_logs_window['log_name'] == 'log.error_count') & (candidate_logs_window['value'] > 0),
    1, 0
)

# Aggregate per (cmdb_id, log_name)
log_agg = candidate_logs_window.groupby(['cmdb_id', 'log_name']).agg(
    window_row_count = ('log_name', 'size'),
    error_count_nonzero_entries = ('error_nonzero', 'sum'),
    min_value_in_window = ('value', 'min'),
    max_value_in_window = ('value', 'max'),
    earliest_timestamp_dt = ('timestamp_dt', 'min')
).reset_index()

# Format earliest timestamp to UTC+8 string
log_agg['earliest_timestamp_in_window_UTC+8'] = log_agg['earliest_timestamp_dt'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
log_agg = log_agg.drop(columns=['earliest_timestamp_dt'])

# Sort and limit to top 20 rows by window_row_count desc, then max_value_in_window desc
log_agg = log_agg.sort_values(by=['window_row_count', 'max_value_in_window'], ascending=[False, False]).head(20)

# 2) For each cmdb_id provide up to 5 example timestamps where log.error_count > 0
examples = []
for cm in candidates:
    subset = candidate_logs_window[
        (candidate_logs_window['cmdb_id'] == cm) &
        (candidate_logs_window['log_name'] == 'log.error_count') &
        (candidate_logs_window['value'] > 0)
    ].sort_values('timestamp_dt')
    if subset.empty:
        ts_str = ''
    else:
        ts_list = subset['timestamp_dt'].dt.strftime('%Y-%m-%dT%H:%M:%S%z').tolist()[:5]
        ts_str = ','.join(ts_list)
    examples.append({'cmdb_id': cm, 'example_log_error_count_gt0_timestamps_UTC+8': ts_str})
examples_df = pd.DataFrame(examples)

# Merge examples into a compact per-cmdb_id summary (keep only candidates that appear in log_agg or all candidates)
# We'll produce a compact table per cmdb_id showing examples; this is not the main returned table but useful to join/display
log_examples_compact = examples_df

# 3) Filter error_logs.csv rows for candidates within window and save for later
candidate_error_logs_window = error_df[
    (error_df['cmdb_id'].isin(candidates)) &
    (error_df['timestamp_dt'] >= window_start) &
    (error_df['timestamp_dt'] < window_end)
].copy()

# Aggregate per candidate cmdb_id: count and earliest up-to-5 timestamps
error_groups = candidate_error_logs_window.groupby('cmdb_id')
error_agg_rows = []
for cm in candidates:
    if cm in error_groups.groups:
        grp = error_groups.get_group(cm).sort_values('timestamp_dt')
        count = len(grp)
        ts_list = grp['timestamp_dt'].dt.strftime('%Y-%m-%dT%H:%M:%S%z').tolist()[:5]
        ts_str = ','.join(ts_list)
    else:
        count = 0
        ts_str = ''
    error_agg_rows.append({
        'cmdb_id': cm,
        'error_logs_count_in_window': count,
        'earliest_up_to_5_timestamps_UTC+8': ts_str
    })
error_agg_df = pd.DataFrame(error_agg_rows)

# Keep displayed outputs compact: log_agg (top 20 rows) and error_agg_df (per candidate)
log_agg_compact = log_agg[['cmdb_id', 'log_name', 'window_row_count', 'error_count_nonzero_entries',
                           'min_value_in_window', 'max_value_in_window', 'earliest_timestamp_in_window_UTC+8']]

# Final returned variables (compact aggregates). Also candidate_logs_window and candidate_error_logs_window are saved in kernel.
log_agg_compact, error_agg_df

```
Out[4]:
```
```python
# Summarize the log and error_log aggregates from the previously computed tables.
# Reuse existing variables: log_agg_compact, error_agg_df

try:
    log_tbl = log_agg_compact.copy()
    err_tbl = error_agg_df.copy()
except NameError:
    # Fallback: load results if not present (should not be necessary in normal flow)
    import pandas as pd
    log_tbl = pd.read_csv('log.csv')  # placeholder fallback (unlikely)
    err_tbl = pd.read_csv('error_logs.csv')

# Prepare summary
candidates = ['IG01','IG02','Tomcat01','Tomcat02','Tomcat03','Tomcat04','MG01','MG02','apache01','apache02']
window_str = "2021-03-09T23:30:00+08:00 through 2021-03-10T00:00:00+08:00"

lines = []
lines.append(f"Candidates: {', '.join(candidates)}")
lines.append(f"Window: {window_str}")
lines.append("")
lines.append("1) Log aggregates (top rows by window_row_count):")
topn = min(5, len(log_tbl))
if topn == 0:
    lines.append("  (no log rows in window for candidates)")
else:
    for _, r in log_tbl.head(20).reset_index(drop=True).iterrows():
        lines.append(
            f"  - {r['cmdb_id']} | {r['log_name']} | rows={int(r['window_row_count'])} | "
            f"errors_nonzero={int(r['error_count_nonzero_entries'])} | "
            f"min={r['min_value_in_window']:.2f} | max={r['max_value_in_window']:.2f} | "
            f"earliest={r['earliest_timestamp_in_window_UTC+8']}"
        )

lines.append("")
lines.append("2) Example timestamps where log.error_count > 0 (up to 5 per cmdb_id):")
# Construct examples from log_tbl/candidate_logs_window if available
try:
    examples_df = log_examples_compact.copy()
except NameError:
    # If not present, try to build minimal examples from candidate_logs_window
    examples_df = None

if examples_df is not None:
    for _, row in examples_df.iterrows():
        ex = row['example_log_error_count_gt0_timestamps_UTC+8']
        ex_display = ex if ex else "(none)"
        lines.append(f"  - {row['cmdb_id']}: {ex_display}")
else:
    lines.append("  (no per-cmdb examples available)")

lines.append("")
lines.append("3) error_logs.csv aggregates (per candidate):")
for _, r in err_tbl.iterrows():
    ts = r['earliest_up_to_5_timestamps_UTC+8']
    ts_display = ts if ts else "(none)"
    lines.append(f"  - {r['cmdb_id']} | error_logs_count_in_window={int(r['error_logs_count_in_window'])} | earliest_up_to_5={ts_display}")

lines.append("")
lines.append("Notes:")
lines.append(" - apache01 and apache02 are the only candidates with non-zero error_log entries in the window (3 each).")
lines.append(" - apache01 and apache02 also show non-zero log.error_count entries in log.csv (apache01: 2 entries, apache02: 3 entries).")
lines.append(" - Other candidates show no error_log entries and no log.error_count > 0 within the window based on the data.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id          log_name  window_row_count  error_count_nonzero_entries  min_value_in_window  max_value_in_window earliest_timestamp_in_window_UTC+8
15  apache02     log.row_count                26                            0                20.00               196.00           2021-03-09T23:30:00+0800
13  apache01     log.row_count                26                            0                21.00               185.00           2021-03-09T23:30:00+0800
12  apache01   log.error_count                26                            2                 0.00                 2.00           2021-03-09T23:30:00+0800
14  apache02   log.error_count                26                            3                 0.00                 1.00           2021-03-09T23:30:00+0800
8   Tomcat03     log.row_count                25                            0                 4.00                98.00           2021-03-09T23:30:00+0800
11  Tomcat04     log.row_count                25                            0                 1.00                93.00           2021-03-09T23:30:00+0800
6   Tomcat03   log.error_count                25                            0                 0.00                 0.00           2021-03-09T23:30:00+0800
9   Tomcat04   log.error_count                25                            0                 0.00                 0.00           2021-03-09T23:30:00+0800
2   Tomcat01     log.row_count                24                            0                 2.00                97.00           2021-03-09T23:30:00+0800
0   Tomcat01   log.error_count                24                            0                 0.00                 0.00           2021-03-09T23:30:00+0800
5   Tomcat02     log.row_count                 6                            0                 1.00                 1.00           2021-03-09T23:31:00+0800
4   Tomcat02  log.gc_pause_max                 6                            0                 0.03                 0.08           2021-03-09T23:31:00+0800
3   Tomcat02   log.error_count                 6                            0                 0.00                 0.00           2021-03-09T23:31:00+0800
10  Tomcat04  log.gc_pause_max                 4                            0                 0.08                 0.10           2021-03-09T23:35:00+0800
1   Tomcat01  log.gc_pause_max                 3                            0                 0.07                 0.10           2021-03-09T23:38:00+0800
7   Tomcat03  log.gc_pause_max                 3                            0                 0.03                 0.08           2021-03-09T23:39:00+0800,     cmdb_id  error_logs_count_in_window                  earliest_up_to_5_timestamps_UTC+8
0      IG01                           0                                                   
1      IG02                           0                                                   
2  Tomcat01                           0                                                   
3  Tomcat02                           0                                                   
4  Tomcat03                           0                                                   
5  Tomcat04                           0                                                   
6      MG01                           0                                                   
7      MG02                           0                                                   
8  apache01                           3  2021-03-09T23:31:04+0800,2021-03-09T23:31:12+0...
9  apache02                           3  2021-03-09T23:30:55+0800,2021-03-09T23:32:02+0...)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import re
import pytz

# Reuse existing in-kernel variables
try:
    trace_anomalies_window
except NameError:
    raise RuntimeError("trace_anomalies_window not found in kernel. Run trace anomaly detection step first.")
try:
    metric_anomalies_window
except NameError:
    raise RuntimeError("metric_anomalies_window not found in kernel. Run metric anomaly detection step first.")

# Ensure timezone
tz = pytz.timezone('Asia/Shanghai')

# Work on copies
ta = trace_anomalies_window.copy()
ma = metric_anomalies_window.copy()

# Ensure timestamp_dt exists and is timezone-aware UTC+8
if 'timestamp_dt' not in ta.columns:
    ta['timestamp_dt'] = pd.to_datetime(ta['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    ta['timestamp_dt'] = pd.to_datetime(ta['timestamp_dt']).dt.tz_convert(tz)

if 'timestamp_dt' not in ma.columns:
    ma['timestamp_dt'] = pd.to_datetime(ma['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    ma['timestamp_dt'] = pd.to_datetime(ma['timestamp_dt']).dt.tz_convert(tz)

# 1) Extract target from trace_name matching trace.to_<Target>. or trace.from_<Target>.
# Use regex to capture the token between 'trace.to_' or 'trace.from_' and the next dot.
ta['extracted_target'] = ta['trace_name'].str.extract(r'trace\.(?:to|from)_([^\.]+)\.', expand=False)

# Filter only rows where extracted_target is present
ta_targets = ta[ta['extracted_target'].notna()].copy()

# 2) Aggregate trace anomalies grouped by (trace source cmdb_id, trace_name, extracted_target)
trace_grp = ta_targets.groupby(['cmdb_id', 'trace_name', 'extracted_target'], dropna=False)

trace_agg = trace_grp.agg(
    trace_global_p95 = ('global_p95', 'first'),
    trace_max_value_in_window = ('value', 'max'),
    trace_earliest_anomaly_timestamp = ('timestamp_dt', 'min'),
    trace_anomaly_count_in_window = ('value', 'size')
).reset_index()

# Ensure we have window_row_count for each (cmdb_id, trace_name) from the original trace window.
# Try to reuse trace_window if present; otherwise build it from trace_df (should exist from prior steps).
try:
    trace_window
except NameError:
    # Reconstruct trace_window from trace_df if available
    try:
        trace_df
    except NameError:
        raise RuntimeError("trace_window and trace_df not available to compute window_row_count. Please run trace window filtering step.")
    # Ensure timestamp_dt in trace_df
    trace_df['timestamp_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    window_start = pd.to_datetime('2021-03-09T23:30:00').tz_localize(tz)
    window_end   = pd.to_datetime('2021-03-10T00:00:00').tz_localize(tz)
    trace_window = trace_df[(trace_df['timestamp_dt'] >= window_start) & (trace_df['timestamp_dt'] < window_end)].copy()

# Compute window_row_count
window_counts = trace_window.groupby(['cmdb_id', 'trace_name']).size().rename('window_row_count').reset_index()

# Merge window_row_count into trace_agg
trace_agg = trace_agg.merge(window_counts, on=['cmdb_id', 'trace_name'], how='left')
trace_agg['window_row_count'] = trace_agg['window_row_count'].fillna(0).astype(int)

# Format trace earliest timestamp to UTC+8 string
trace_agg['trace_earliest_anomaly_timestamp_UTC+8'] = trace_agg['trace_earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
trace_agg = trace_agg.drop(columns=['trace_earliest_anomaly_timestamp'])

# 3) For each extracted target, analyze metric anomalies in metric_anomalies_window
# Build per-target metric KPI summary: per KPI max in window and global_p95, then severity ratio, then per-target max ratio and earliest timestamp
if ma.empty:
    per_target_metrics = pd.DataFrame(columns=['cmdb_id','target_has_metric_anomaly','target_earliest_metric_anomaly_timestamp_UTC+8','target_max_metric_severity_ratio'])
else:
    per_kpi = ma.groupby(['cmdb_id','kpi_name']).agg(
        kpi_max_value_in_window = ('value','max'),
        kpi_global_p95 = ('global_p95','first'),
        kpi_earliest_ts = ('timestamp_dt','min')
    ).reset_index()
    # severity ratio per KPI
    per_kpi['severity_ratio'] = per_kpi['kpi_max_value_in_window'] / per_kpi['kpi_global_p95']
    # earliest metric anomaly per target
    earliest_metric_ts = ma.groupby('cmdb_id')['timestamp_dt'].min().reset_index().rename(columns={'timestamp_dt':'earliest_metric_ts'})
    earliest_metric_ts['earliest_metric_ts_str'] = earliest_metric_ts['earliest_metric_ts'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
    # per-target max severity ratio
    per_target_ratio = per_kpi.groupby('cmdb_id')['severity_ratio'].max().reset_index().rename(columns={'severity_ratio':'target_max_metric_severity_ratio'})
    # merge earliest ts and ratio
    per_target_metrics = per_target_ratio.merge(earliest_metric_ts[['cmdb_id','earliest_metric_ts_str']], on='cmdb_id', how='left')
    per_target_metrics = per_target_metrics.rename(columns={
        'cmdb_id':'cmdb_id',
        'earliest_metric_ts_str':'target_earliest_metric_anomaly_timestamp_UTC+8'
    })
    per_target_metrics['target_has_metric_anomaly'] = True

# For targets without anomalies, ensure entries will show False/NaN
# Merge trace_agg (which has extracted_target) with per_target_metrics by matching extracted_target -> cmdb_id
trace_agg = trace_agg.merge(per_target_metrics, left_on='extracted_target', right_on='cmdb_id', how='left', suffixes=('','_metric'))

# Fill missing metric flags
trace_agg['target_has_metric_anomaly'] = trace_agg['target_has_metric_anomaly'].fillna(False)
# target_earliest_metric_anomaly_timestamp_UTC+8 may be NaN; keep as is
# target_max_metric_severity_ratio may be missing; keep NaN or set 0 per instruction (use NaN to indicate none)
if 'target_max_metric_severity_ratio' not in trace_agg.columns:
    trace_agg['target_max_metric_severity_ratio'] = np.nan

# 4) Build final correlated table with requested columns
trace_metric_correlation = trace_agg[[
    'cmdb_id',                      # trace source cmdb_id
    'trace_name',
    'extracted_target',
    'trace_global_p95',
    'trace_max_value_in_window',
    'trace_earliest_anomaly_timestamp_UTC+8',
    'trace_anomaly_count_in_window',
    'target_has_metric_anomaly',
    'target_earliest_metric_anomaly_timestamp_UTC+8',
    'target_max_metric_severity_ratio'
]].rename(columns={
    'cmdb_id':'trace_source_cmdb_id',
    'trace_name':'trace_name',
    'extracted_target':'extracted_target',
    'trace_global_p95':'trace_global_p95',
    'trace_max_value_in_window':'trace_max_value_in_window',
    'trace_earliest_anomaly_timestamp_UTC+8':'trace_earliest_anomaly_timestamp_UTC+8',
    'trace_anomaly_count_in_window':'trace_anomaly_count_in_window'
})

# 5) Sort by target_has_metric_anomaly desc, then target_max_metric_severity_ratio desc, then trace_max_value_in_window desc.
# For sorting, replace NaN severity with 0 in a helper column to allow numeric sort but keep has_metric flag primary.
trace_metric_correlation['metric_severity_sort'] = trace_metric_correlation['target_max_metric_severity_ratio'].fillna(0.0)
trace_metric_correlation = trace_metric_correlation.sort_values(
    by=['target_has_metric_anomaly', 'metric_severity_sort', 'trace_max_value_in_window'],
    ascending=[False, False, False]
).drop(columns=['metric_severity_sort'])

# 6) Limit to top 20 rows and save into kernel variable trace_metric_correlation
trace_metric_correlation = trace_metric_correlation.head(20)

# Final compact table (as DataFrame)
trace_metric_correlation

```
Out[5]:
```
```python
# Summarize the trace->metric correlation results in plain English using the existing trace_metric_correlation variable.

try:
    df = trace_metric_correlation.copy()
except NameError:
    raise RuntimeError("trace_metric_correlation not found in kernel. Run the correlation step first.")

num_rows = int(df.shape[0])
num_targets_with_metric = int(df['target_has_metric_anomaly'].sum())
distinct_targets = df['extracted_target'].dropna().unique().tolist()
earliest_trace_anom = df['trace_earliest_anomaly_timestamp_UTC+8'].min() if num_rows > 0 else "N/A"

# Prepare top 5 example lines
examples = []
for _, r in df.head(5).iterrows():
    examples.append(
        f"{r['trace_source_cmdb_id']} -> {r['extracted_target']} | {r['trace_name']} | "
        f"trace_max={float(r['trace_max_value_in_window']):.6g} | trace_p95={float(r['trace_global_p95']):.6g} | "
        f"trace_anoms={int(r['trace_anomaly_count_in_window'])} | target_has_metric_anom={bool(r['target_has_metric_anomaly'])} | "
        f"target_max_severity={('NaN' if pd.isna(r['target_max_metric_severity_ratio']) else f'{float(r['target_max_metric_severity_ratio']):.6g}')}"
    )

summary_lines = [
    "Correlation summary between trace anomalies and metric anomalies (compact):",
    f"- Total correlated trace->target entries returned: {num_rows}",
    f"- Distinct extracted targets observed (examples): {distinct_targets}",
    f"- Number of entries whose extracted target has metric anomalies: {num_targets_with_metric}",
    f"- Earliest trace anomaly timestamp (UTC+8) in these entries: {earliest_trace_anom}",
    "",
    "Top example correlations (up to 5):"
]
summary_lines += [f"  - {e}" for e in examples]

# Notable observation if severity ratios present and high
if 'target_max_metric_severity_ratio' in df.columns and df['target_max_metric_severity_ratio'].notna().any():
    max_sev = df['target_max_metric_severity_ratio'].max()
    max_sev_row = df.loc[df['target_max_metric_severity_ratio'].idxmax()]
    summary_lines.append("")
    summary_lines.append(
        f"Notable: highest target_max_metric_severity_ratio = {float(max_sev):.6g} for target "
        f"{max_sev_row['extracted_target']} (trace source {max_sev_row['trace_source_cmdb_id']})."
    )

summary_lines.append("")
summary_lines.append("Interpretation:")
summary_lines.append("- Many trace anomalies map to the target component 'MG01'.")
summary_lines.append("- For those targets, metric anomalies exist (target_has_metric_anomaly=True) and show high severity ratios (e.g., ~20x),")
summary_lines.append("  indicating strong metric breaches on the downstream target around the trace anomaly times.")
summary_lines.append("- These correlated entries are priority candidates for investigation (start with MG01 and its metric KPIs).")

import pandas as pd
summary = "\n".join(summary_lines)

# Display the plain-English summary
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

trace_source_cmdb_id                     trace_name extracted_target  trace_global_p95  trace_max_value_in_window trace_earliest_anomaly_timestamp_UTC+8  trace_anomaly_count_in_window  target_has_metric_anomaly target_earliest_metric_anomaly_timestamp_UTC+8  target_max_metric_severity_ratio
151             dockerA1   trace.from_MG01.duration_p95             MG01        179.387505                 202.881200               2021-03-09T23:30:00+0800                              2                       True                       2021-03-09T23:30:00+0800                              20.0
163             dockerA2   trace.from_MG01.duration_p95             MG01        159.909025                 202.427300               2021-03-09T23:30:00+0800                              2                       True                       2021-03-09T23:30:00+0800                              20.0
150             dockerA1  trace.from_MG01.duration_mean             MG01        159.590750                 173.894000               2021-03-09T23:30:00+0800                              2                       True                       2021-03-09T23:30:00+0800                              20.0
175             dockerB1   trace.from_MG01.duration_p95             MG01         13.285260                 163.912250               2021-03-09T23:32:00+0800                              2                       True                       2021-03-09T23:30:00+0800                              20.0
162             dockerA2  trace.from_MG01.duration_mean             MG01         75.390181                 146.915000               2021-03-09T23:32:00+0800                              2                       True                       2021-03-09T23:30:00+0800                              20.0
182             dockerB1        trace.to_MG01.row_count             MG01         67.200000                  81.000000               2021-03-09T23:42:00+0800                              2                       True                       2021-03-09T23:30:00+0800                              20.0
174             dockerB1  trace.from_MG01.duration_mean             MG01         12.755919                  77.581667               2021-03-09T23:32:00+0800                              2                       True                       2021-03-09T23:30:00+0800                              20.0
122             Tomcat02        trace.to_MG01.row_count             MG01         56.400000                  76.000000               2021-03-09T23:56:00+0800                              2                       True                       2021-03-09T23:30:00+0800                              20.0
110             Tomcat01        trace.to_MG01.row_count             MG01         57.500000                  69.000000               2021-03-09T23:58:00+0800                              2                       True                       2021-03-09T23:30:00+0800                              20.0
188             dockerB2      trace.from_MG01.row_count             MG01         53.750000                  69.000000               2021-03-09T23:49:00+0800                              2                       True                       2021-03-09T23:30:00+0800                              20.0
197             dockerB2        trace.to_MG01.row_count             MG01         58.400000                  68.000000               2021-03-09T23:41:00+0800                              2                       True                       2021-03-09T23:30:00+0800                              20.0
146             Tomcat04        trace.to_MG01.row_count             MG01         49.000000                  66.000000               2021-03-09T23:49:00+0800                              2                       True                       2021-03-09T23:30:00+0800                              20.0
134             Tomcat03        trace.to_MG01.row_count             MG01         55.800000                  65.000000               2021-03-09T23:56:00+0800                              2                       True                       2021-03-09T23:30:00+0800                              20.0
176             dockerB1      trace.from_MG01.row_count             MG01         54.400000                  62.000000               2021-03-09T23:42:00+0800                              2                       True                       2021-03-09T23:30:00+0800                              20.0
145             Tomcat04     trace.to_MG01.duration_p95             MG01         52.863590                  60.002600               2021-03-09T23:32:00+0800                              2                       True                       2021-03-09T23:30:00+0800                              20.0
121             Tomcat02     trace.to_MG01.duration_p95             MG01         24.913610                  60.002000               2021-03-09T23:30:00+0800                              2                       True                       2021-03-09T23:30:00+0800                              20.0
109             Tomcat01     trace.to_MG01.duration_p95             MG01         54.492822                  60.001750               2021-03-09T23:32:00+0800                              2                       True                       2021-03-09T23:30:00+0800                              20.0
133             Tomcat03     trace.to_MG01.duration_p95             MG01         38.810440                  60.001000               2021-03-09T23:30:00+0800                              2                       True                       2021-03-09T23:30:00+0800                              20.0
164             dockerA2      trace.from_MG01.row_count             MG01         36.100000                  54.000000               2021-03-09T23:48:00+0800                              2                       True                       2021-03-09T23:30:00+0800                              20.0
152             dockerA1      trace.from_MG01.row_count             MG01         43.650000                  48.000000               2021-03-09T23:52:00+0800                              2                       True                       2021-03-09T23:30:00+0800                              20.0```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing dataframes if available, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')

try:
    error_df
except NameError:
    error_df = pd.read_csv('error_logs.csv')

# Ensure numeric 'value' in metric and set timezone
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 across the full metric.csv (rule: compute BEFORE window filtering)
global_p95 = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).reset_index().rename(columns={'value':'global_p95'})

# Merge global_p95 back to full metric_df to ensure availability
metric_df = metric_df.merge(global_p95, on=['cmdb_id','kpi_name'], how='left')

# Ensure timestamp_dt column (timezone-aware UTC+8)
metric_df['timestamp_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Define failure window (inclusive start, exclusive end)
window_start = pd.to_datetime('2021-03-09T23:30:00').tz_localize(tz)
window_end   = pd.to_datetime('2021-03-10T00:00:00').tz_localize(tz)

# Filter metric_df to window
metric_window = metric_df[(metric_df['timestamp_dt'] >= window_start) & (metric_df['timestamp_dt'] < window_end)].copy()

# Detect metric anomalies within window: value >= global_p95
metric_window['is_anomaly'] = metric_window['value'] >= metric_window['global_p95']

# Filter MG01 rows within window (both all window rows and anomaly rows)
mg01_metric_window = metric_window[metric_window['cmdb_id'] == 'MG01'].copy()
mg01_metric_anomalies_window = mg01_metric_window[mg01_metric_window['is_anomaly']].copy()

# Helper to compute longest consecutive anomalous minutes for a kpi within the window
def longest_consecutive_anomalous_minutes_for_grp(df_grp):
    # df_grp must contain timestamp_dt and is_anomaly columns, sorted by timestamp_dt
    if df_grp.empty:
        return 0
    df_grp = df_grp.sort_values('timestamp_dt')
    # Select only anomalous rows
    anom = df_grp[df_grp['is_anomaly']]
    if anom.empty:
        return 0
    # Compute seconds since epoch for anomalies
    # Use view to get int64 nanoseconds
    secs = (anom['timestamp_dt'].view('int64') // 10**9).to_numpy()
    # Count consecutive runs where difference == 60 seconds
    max_run = 1
    current = 1
    for i in range(1, len(secs)):
        if secs[i] - secs[i-1] == 60:
            current += 1
        else:
            if current > max_run:
                max_run = current
            current = 1
    if current > max_run:
        max_run = current
    return int(max_run)

# 1) Aggregate per MG01 kpi_name
rows = []
kpi_groups = mg01_metric_window.groupby('kpi_name')
for kpi, grp in kpi_groups:
    gp95 = grp['global_p95'].iloc[0]  # global p95 computed from full file
    window_row_count = len(grp)
    # anomaly rows for this kpi in window
    grp_anom = grp[grp['is_anomaly']]
    anomaly_count = len(grp_anom)
    earliest_anom_ts = grp_anom['timestamp_dt'].min() if anomaly_count>0 else pd.NaT
    earliest_anom_str = earliest_anom_ts.strftime('%Y-%m-%dT%H:%M:%S%z') if pd.notna(earliest_anom_ts) else ''
    max_value = grp['value'].max() if window_row_count>0 else np.nan
    severity_ratio = (max_value / gp95) if (pd.notna(max_value) and pd.notna(gp95) and gp95!=0) else np.nan
    longest_run = longest_consecutive_anomalous_minutes_for_grp(grp)
    rows.append({
        'kpi_name': kpi,
        'global_p95': float(gp95) if pd.notna(gp95) else np.nan,
        'anomaly_count_in_window': int(anomaly_count),
        'earliest_anomaly_timestamp_UTC+8': earliest_anom_str,
        'max_value_in_window': float(max_value) if pd.notna(max_value) else np.nan,
        'window_row_count': int(window_row_count),
        'severity_ratio': float(severity_ratio) if pd.notna(severity_ratio) else np.nan,
        'longest_consecutive_anomalous_minutes': int(longest_run)
    })

mg01_kpi_agg = pd.DataFrame(rows)

# Sort by severity desc and limit to top 20
mg01_kpi_agg = mg01_kpi_agg.sort_values(by='severity_ratio', ascending=False).head(20).reset_index(drop=True)

# 2) Flag likely true faults: severity_ratio > 1.5 and longest_consecutive_anomalous_minutes >= 2
mg01_flagged = mg01_kpi_agg[
    (mg01_kpi_agg['severity_ratio'] > 1.5) &
    (mg01_kpi_agg['longest_consecutive_anomalous_minutes'] >= 2)
].head(20).reset_index(drop=True)

# 3) Logs for MG01 in the window
log_df['timestamp_dt'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
mg01_logs_window = log_df[(log_df['cmdb_id'] == 'MG01') & (log_df['timestamp_dt'] >= window_start) & (log_df['timestamp_dt'] < window_end)].copy()

total_log_rows = len(mg01_logs_window)
distinct_log_names = mg01_logs_window['log_name'].dropna().unique().tolist()[:10]
earliest_log_ts = mg01_logs_window['timestamp_dt'].min()
latest_log_ts = mg01_logs_window['timestamp_dt'].max()
earliest_log_str = earliest_log_ts.strftime('%Y-%m-%dT%H:%M:%S%z') if pd.notna(earliest_log_ts) else ''
latest_log_str = latest_log_ts.strftime('%Y-%m-%dT%H:%M:%S%z') if pd.notna(latest_log_ts) else ''

mg01_log_agg = pd.DataFrame([{
    'cmdb_id': 'MG01',
    'total_log_rows_in_window': int(total_log_rows),
    'list_of_log_name_values_present_up_to_10': ','.join(distinct_log_names),
    'earliest_timestamp_in_window_UTC+8': earliest_log_str,
    'latest_timestamp_in_window_UTC+8': latest_log_str
}])

# Up to 10 raw error_logs rows for MG01 within the window (examples)
error_df['timestamp_dt'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
mg01_error_logs_window = error_df[(error_df['cmdb_id'] == 'MG01') & (error_df['timestamp_dt'] >= window_start) & (error_df['timestamp_dt'] < window_end)].copy()
mg01_error_examples = mg01_error_logs_window[['timestamp', 'cmdb_id', 'message']].sort_values('timestamp').head(10)

# Save the filtered MG01 metric anomaly rows and logs to kernel variables for later inspection
# mg01_metric_anomalies_window : anomaly rows for MG01 (value >= global_p95)
mg01_metric_anomalies_window = mg01_metric_anomalies_window if 'mg01_metric_anomalies_window' in globals() else mg01_metric_anomalies_window
mg01_metric_anomalies_window = mg01_metric_anomalies_window  # already defined above

mg01_logs_window = mg01_logs_window  # already defined above

# Return compact tables: mg01_kpi_agg, mg01_flagged, mg01_log_agg, mg01_error_examples
mg01_kpi_agg, mg01_flagged, mg01_log_agg, mg01_error_examples

```
Out[6]:
```
```python
# Summarize findings for MG01 using in-kernel variables: mg01_kpi_agg, mg01_flagged, mg01_log_agg, mg01_error_examples
try:
    mg_kpi = mg01_kpi_agg.copy()
    mg_flagged = mg01_flagged.copy()
    mg_log_agg = mg01_log_agg.copy()
    mg_err_examples = mg01_error_examples.copy()
except NameError:
    raise RuntimeError("Expected MG01 result variables not found in kernel.")

lines = []
lines.append("MG01 investigation summary for window 2021-03-09T23:30:00+08 to 2021-03-10T00:00:00+08:")
lines.append("")
# KPI anomalies summary
num_kpis = mg_kpi.shape[0]
lines.append(f"- Number of MG01 KPI entries with anomalies reported (top list returned): {num_kpis}")
if num_kpis > 0:
    lines.append("- Top KPI anomaly examples (kpi_name | severity_ratio | longest_consecutive_minutes | earliest_anomaly | max_value):")
    for _, r in mg_kpi.head(5).iterrows():
        lines.append(
            f"  - {r['kpi_name']} | severity={r['severity_ratio']:.4g} | longest_run={int(r['longest_consecutive_anomalous_minutes'])}m | "
            f"earliest={r['earliest_anomaly_timestamp_UTC+8'] or '(none)'} | max={r['max_value_in_window']:.4g}"
        )
else:
    lines.append("  (no metric anomalies for MG01 in the window)")

# Flagged likely true faults
num_flagged = mg_flagged.shape[0]
lines.append("")
lines.append(f"- Flagged likely true faults (severity_ratio>1.5 AND longest_consecutive_anomalous_minutes>=2): {num_flagged}")
if num_flagged > 0:
    lines.append("  Flagged KPI rows:")
    for _, r in mg_flagged.iterrows():
        lines.append(
            f"  - {r['kpi_name']} | severity={r['severity_ratio']:.4g} | longest_run={int(r['longest_consecutive_anomalous_minutes'])}m | earliest={r['earliest_anomaly_timestamp_UTC+8']}"
        )
else:
    lines.append("  No MG01 KPI met both severity and consecutive-minute thresholds in this window.")

# Logs summary
lines.append("")
total_logs = int(mg_log_agg.at[0, 'total_log_rows_in_window']) if (not mg_log_agg.empty and 'total_log_rows_in_window' in mg_log_agg.columns) else 0
lines.append(f"- MG01 logs in window: total_log_rows_in_window = {total_logs}")
if total_logs > 0:
    names = mg_log_agg.at[0, 'list_of_log_name_values_present_up_to_10']
    earliest = mg_log_agg.at[0, 'earliest_timestamp_in_window_UTC+8']
    latest = mg_log_agg.at[0, 'latest_timestamp_in_window_UTC+8']
    lines.append(f"  - log names (up to 10): {names}")
    lines.append(f"  - earliest log timestamp: {earliest}")
    lines.append(f"  - latest log timestamp: {latest}")
else:
    lines.append("  - No log rows for MG01 in the window.")

# Error log examples
err_count = 0 if mg_err_examples.empty else len(mg_err_examples)
lines.append("")
lines.append(f"- error_logs examples for MG01 in window: {err_count} rows returned (up to 10 shown).")
if err_count > 0:
    lines.append("  Example error_log timestamps/messages:")
    for _, r in mg_err_examples.iterrows():
        # convert integer timestamp to UTC+8 string if needed
        try:
            ts = pd.to_datetime(int(r['timestamp']), unit='s', utc=True).tz_convert(pytz.timezone('Asia/Shanghai')).strftime('%Y-%m-%dT%H:%M:%S%z')
        except Exception:
            ts = str(r['timestamp'])
        lines.append(f"  - {ts} | message={r.get('message','')}")
else:
    lines.append("  - No error_log rows for MG01 in the window.")

lines.append("")
lines.append("Key interpretation:")
lines.append("- Multiple MG01 KPIs breached their global P95 (severity ratios >1.0), with the highest severity observed on network/CPU/disk-related KPIs.")
lines.append("- However, none met the stricter fault criteria (severity >1.5 AND consecutive >=2 minutes), so there are no strongly sustained metric faults by this rule.")
lines.append("- There are no MG01 application logs or error_logs in the window to support/confirm metric findings (no log rows and no error_log examples).")
lines.append("- Recommendation: investigate top-severity KPIs (e.g., 'OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT', 'OSLinux-CPU_CPU_CPUWio', 'OSLinux-CPU_CPU-3_SingleCpuidle', and disk KPIs) for transient spikes; correlate with surrounding minutes and traces into/out of MG01.")

summary = "\n".join(lines)

# Display the summary string
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                             kpi_name    global_p95  anomaly_count_in_window earliest_anomaly_timestamp_UTC+8  max_value_in_window  window_row_count  severity_ratio  longest_consecutive_anomalous_minutes
0      OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  5.000000e-02                        1         2021-03-09T23:56:00+0800         1.000000e+00                20       20.000000                                      1
1                              OSLinux-CPU_CPU_CPUWio  7.261600e-01                        1         2021-03-09T23:55:00+0800         5.676800e+00                20        7.817561                                      1
2                     OSLinux-CPU_CPU-3_SingleCpuidle  1.401961e+01                        1         2021-03-09T23:30:00+0800         2.549020e+01                10        1.818182                                      1
3      OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps  1.650000e+00                        1         2021-03-09T23:55:00+0800         3.000000e+00                10        1.818182                                      1
4      OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps  2.200000e+00                        1         2021-03-09T23:40:00+0800         4.000000e+00                10        1.818182                                      1
5   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...  4.100000e+01                        1         2021-03-09T23:55:00+0800         7.100000e+01                11        1.731707                                      1
6                     OSLinux-CPU_CPU-2_SingleCpuUtil  1.637642e+01                        1         2021-03-09T23:32:00+0800         2.599730e+01                 9        1.587484                                      1
7        OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  4.100000e+00                        1         2021-03-09T23:33:00+0800         6.000000e+00                20        1.463415                                      1
8      OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps  4.500000e+00                        1         2021-03-09T23:59:00+0800         6.000000e+00                 7        1.333333                                      1
9   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...  3.400000e-01                        1         2021-03-09T23:59:00+0800         4.000000e-01                 7        1.176471                                      1
10  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...  2.650000e-01                        1         2021-03-09T23:38:00+0800         3.000000e-01                 8        1.132075                                      1
11                    OSLinux-CPU_CPU-1_SingleCpuUtil  3.070900e+00                        1         2021-03-09T23:54:00+0800         3.459700e+00                10        1.126608                                      1
12    OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec  1.749653e+05                        1         2021-03-09T23:41:00+0800         1.947640e+05                10        1.113158                                      1
13                            OSLinux-CPU_CPU_CPULoad  1.186500e+00                        1         2021-03-09T23:38:00+0800         1.310000e+00                20        1.104088                                      1
14    OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum  3.954000e+02                        1         2021-03-09T23:35:00+0800         4.350000e+02                 4        1.100152                                      1
15   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite  1.319335e+01                        1         2021-03-09T23:45:00+0800         1.426670e+01                 8        1.081355                                      1
16         JVM-Memory_7779_JVM_Memory_HeapMemoryUsage  4.793055e+01                        1         2021-03-09T23:58:00+0800         5.153460e+01                19        1.075193                                      1
17    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps  1.019145e+00                        1         2021-03-09T23:45:00+0800         1.083300e+00                 8        1.062950                                      1
18                         OSLinux-CPU_CPU_CPUSysTime  5.670750e-01                        1         2021-03-09T23:55:00+0800         5.989000e-01                20        1.056121                                      1
19          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  1.014612e+09                        2         2021-03-09T23:39:00+0800         1.055284e+09                30        1.040086                                      1, Empty DataFrame
Columns: [kpi_name, global_p95, anomaly_count_in_window, earliest_anomaly_timestamp_UTC+8, max_value_in_window, window_row_count, severity_ratio, longest_consecutive_anomalous_minutes]
Index: [],   cmdb_id  total_log_rows_in_window list_of_log_name_values_present_up_to_10 earliest_timestamp_in_window_UTC+8 latest_timestamp_in_window_UTC+8
0    MG01                         0                                                                                                             , Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Reuse in-kernel variables where possible
try:
    metric_anomalies_window
except NameError:
    metric_anomalies_window = None

try:
    global_p95
except NameError:
    # compute global_p95 from full metric.csv if missing
    metric_df = pd.read_csv('metric.csv')
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
    global_p95 = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).reset_index().rename(columns={'value':'global_p95'})

# Load metric full series if needed
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Ensure timestamp_dt in metric_df (UTC -> convert to Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
metric_df['timestamp_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Define failure window
window_start = pd.to_datetime('2021-03-09T23:30:00').tz_localize(tz)
window_end   = pd.to_datetime('2021-03-10T00:00:00').tz_localize(tz)
window_row_count_expected = int((window_end - window_start).total_seconds() / 60)

# Build metric_window (all metric rows within window)
metric_window = metric_df[(metric_df['timestamp_dt'] >= window_start) & (metric_df['timestamp_dt'] < window_end)].copy()

# Ensure metric_anomalies_window: if absent, compute from metric_window using global_p95
if metric_anomalies_window is None:
    # merge global_p95 to metric_window
    metric_window = metric_window.merge(global_p95, on=['cmdb_id','kpi_name'], how='left')
    metric_window['is_anomaly'] = metric_window['value'] >= metric_window['global_p95']
    metric_anomalies_window = metric_window[metric_window['is_anomaly']].copy()
else:
    # ensure timestamp_dt exists and is tz-aware
    if 'timestamp_dt' not in metric_anomalies_window.columns:
        metric_anomalies_window['timestamp_dt'] = pd.to_datetime(metric_anomalies_window['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    # make sure metric_window has global_p95 merged for later per-kpi stats
    if 'global_p95' not in metric_window.columns:
        metric_window = metric_window.merge(global_p95, on=['cmdb_id','kpi_name'], how='left')

# Candidate components
candidates = ['IG01','IG02','MG01','MG02','Tomcat01','Tomcat02','Tomcat03','Tomcat04','apache01','apache02']

# Per-kpi in-window stats (window_row_count, max_value_in_window, global_p95)
per_kpi_window = metric_window.groupby(['cmdb_id','kpi_name']).agg(
    window_row_count = ('value','size'),
    max_value_in_window = ('value','max'),
    # keep first global_p95 (should be same for group)
    global_p95 = ('global_p95','first')
).reset_index()

# Per-kpi anomaly counts and earliest anomaly timestamp (from metric_anomalies_window)
if metric_anomalies_window.empty:
    per_kpi_anom = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count_in_window','earliest_anomaly_ts'])
else:
    per_kpi_anom = metric_anomalies_window.groupby(['cmdb_id','kpi_name']).agg(
        anomaly_count_in_window = ('value','size'),
        earliest_anomaly_ts = ('timestamp_dt','min')
    ).reset_index()

# Merge per-kpi stats
per_kpi = per_kpi_window.merge(per_kpi_anom, on=['cmdb_id','kpi_name'], how='left')
per_kpi['anomaly_count_in_window'] = per_kpi['anomaly_count_in_window'].fillna(0).astype(int)
per_kpi['earliest_anomaly_timestamp_UTC+8'] = per_kpi['earliest_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
per_kpi.loc[per_kpi['earliest_anomaly_timestamp_UTC+8'] == 'NaT', 'earliest_anomaly_timestamp_UTC+8'] = ''
per_kpi = per_kpi.drop(columns=['earliest_anomaly_ts'])

# severity ratio per kpi
per_kpi['severity_ratio'] = per_kpi['max_value_in_window'] / per_kpi['global_p95']
per_kpi['severity_ratio'] = per_kpi['severity_ratio'].replace([np.inf, -np.inf], np.nan)

# Compute longest consecutive anomalous minutes per kpi within window
# Prepare a helper series of anomalies per (cmdb_id,kpi_name)
metric_window['is_anomaly'] = metric_window['value'] >= metric_window['global_p95']
longest_runs = []
grp = metric_window.sort_values('timestamp_dt').groupby(['cmdb_id','kpi_name'])
for (cm, kpi), g in grp:
    anom = g[g['is_anomaly']].sort_values('timestamp_dt')
    if anom.empty:
        longest = 0
    else:
        secs = (anom['timestamp_dt'].view('int64') // 10**9).to_numpy()
        max_run = 1
        cur = 1
        for i in range(1, len(secs)):
            if secs[i] - secs[i-1] == 60:
                cur += 1
            else:
                if cur > max_run:
                    max_run = cur
                cur = 1
        if cur > max_run:
            max_run = cur
        longest = int(max_run)
    longest_runs.append({'cmdb_id':cm, 'kpi_name':kpi, 'longest_consecutive_anomalous_minutes':longest})
longest_df = pd.DataFrame(longest_runs)

# Merge longest runs into per_kpi
per_kpi = per_kpi.merge(longest_df, on=['cmdb_id','kpi_name'], how='left')
per_kpi['longest_consecutive_anomalous_minutes'] = per_kpi['longest_consecutive_anomalous_minutes'].fillna(0).astype(int)

# Now aggregate per component for the candidate list
comp_rows = []
for comp in candidates:
    comp_kpis = per_kpi[per_kpi['cmdb_id'] == comp].copy()
    total_distinct_kpis_with_anomalies_in_window = int((comp_kpis['anomaly_count_in_window']>0).sum())
    total_anomaly_points_in_window = int(comp_kpis['anomaly_count_in_window'].sum())
    # sustained full window: anomaly_count == window_row_count (for that kpi)
    num_kpis_sustained_full_window = int(((comp_kpis['anomaly_count_in_window'] == comp_kpis['window_row_count']) & (comp_kpis['window_row_count']>0)).sum())
    # max severity ratio across kpis
    if not comp_kpis.empty and comp_kpis['severity_ratio'].notna().any():
        max_sev = float(comp_kpis['severity_ratio'].replace([np.inf, -np.inf], np.nan).max())
    else:
        max_sev = 0.0
    # earliest anomaly timestamp across component
    earliest_ts = ''
    if comp_kpis['earliest_anomaly_timestamp_UTC+8'].replace('', pd.NA).dropna().shape[0] > 0:
        valid_ts = comp_kpis.loc[comp_kpis['earliest_anomaly_timestamp_UTC+8']!='','earliest_anomaly_timestamp_UTC+8']
        if not valid_ts.empty:
            earliest_ts = valid_ts.min()
    # window_row_count_expected (same for all components)
    expected = window_row_count_expected
    # has_log_entries_in_window: check log.csv
    try:
        candidate_logs_window
    except NameError:
        # compute logs in window
        log_df = pd.read_csv('log.csv')
        log_df['timestamp_dt'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        candidate_logs_window = log_df[(log_df['cmdb_id'].isin(candidates)) & (log_df['timestamp_dt'] >= window_start) & (log_df['timestamp_dt'] < window_end)].copy()
    has_logs = bool(((candidate_logs_window['cmdb_id'] == comp)).any())
    # error_logs_count_in_window
    try:
        candidate_error_logs_window
    except NameError:
        error_df = pd.read_csv('error_logs.csv')
        error_df['timestamp_dt'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        candidate_error_logs_window = error_df[(error_df['cmdb_id'].isin(candidates)) & (error_df['timestamp_dt'] >= window_start) & (error_df['timestamp_dt'] < window_end)].copy()
    err_count = int(((candidate_error_logs_window['cmdb_id'] == comp)).sum())
    comp_rows.append({
        'cmdb_id': comp,
        'total_distinct_kpis_with_anomalies_in_window': total_distinct_kpis_with_anomalies_in_window,
        'total_anomaly_points_in_window': total_anomaly_points_in_window,
        'num_kpis_sustained_full_window': num_kpis_sustained_full_window,
        'max_severity_ratio_across_kpis': max_sev,
        'earliest_anomaly_timestamp_UTC+8': earliest_ts,
        'window_row_count_expected': expected,
        'has_log_entries_in_window': bool(has_logs),
        'error_logs_count_in_window': err_count
    })

candidate_component_summary = pd.DataFrame(comp_rows)

# Sort for readability: by total_anomaly_points desc, then max_severity_ratio desc
candidate_component_summary = candidate_component_summary.sort_values(
    by=['total_anomaly_points_in_window','max_severity_ratio_across_kpis'],
    ascending=[False, False]
).reset_index(drop=True)

# Limit to top 20 rows (candidates list is <=20)
candidate_component_summary = candidate_component_summary.head(20)

# Return the compact aggregate table
candidate_component_summary

```
Out[7]:
```
```python
# Plain-English summary of the candidate_component_summary results
summary = (
    "Compact comparison (window 2021-03-09T23:30:00+08 to 2021-03-10T00:00:00+08):\n"
    "- Top components by anomaly point volume: Tomcat02 (1009 points), Tomcat03 (955), Tomcat04 (933), Tomcat01 (855).\n"
    "- Many Tomcat instances show large numbers of distinct KPIs with anomalies (≈107–111 distinct KPIs) and many KPIs sustained every minute (30-min window: 52–67 KPIs breached for every minute).\n"
    "- IG01 and IG02 also show substantial anomaly activity (≈90 distinct KPIs, ~610 anomalies each) with high per-KPI severity (IG01 max severity ~11.7, IG02 ~8.7).\n"
    "- MG01 and MG02 show many KPI breaches (~92 distinct KPIs each). MG01 has an especially high max severity ratio (20.0) indicating a KPI spike ~20x above its global P95.\n"
    "- apache01 and apache02 have fewer distinct KPI anomalies (~82 each) but very high peak severity ratios (apache01 ~25.8, apache02 ~14.3). Both apache nodes have log entries in the window and 3 error_log rows each.\n"
    "- Earliest anomaly across components is at 2021-03-09T23:30:00+08 for many components. Expected window length = 30 minutes.\n"
    "- Log presence: Tomcat* and apache* have log entries in-window (True). IG* and MG* show no log entries in-window in the filtered logs.\n"
    "Interpretation / next steps:\n"
    " - Prioritize investigation on Tomcat02/03/04/01 for broad, sustained KPI breaches (many KPIs breached every minute).\n"
    " - Immediately inspect apache01 (highest severity ratio) and MG01 (very high severity ratio) for root-cause signals; apache nodes also have error logs to examine.\n"
    " - Correlate the top-severity KPI spikes (apache01, MG01, IG01) with traces and any available logs around the earliest anomaly times (≈23:30 UTC+8).\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  total_distinct_kpis_with_anomalies_in_window  total_anomaly_points_in_window  num_kpis_sustained_full_window  max_severity_ratio_across_kpis earliest_anomaly_timestamp_UTC+8  window_row_count_expected  has_log_entries_in_window  error_logs_count_in_window
0  Tomcat02                                           111                            1009                              67                        4.575424         2021-03-09T23:30:00+0800                         30                       True                           0
1  Tomcat03                                           109                             955                              57                        5.813004         2021-03-09T23:30:00+0800                         30                       True                           0
2  Tomcat04                                           109                             933                              52                        3.714286         2021-03-09T23:30:00+0800                         30                       True                           0
3  Tomcat01                                           107                             855                              56                        2.431988         2021-03-09T23:30:00+0800                         30                       True                           0
4      IG01                                            90                             612                              36                       11.739659         2021-03-09T23:30:00+0800                         30                      False                           0
5      IG02                                            90                             610                              35                        8.671824         2021-03-09T23:30:00+0800                         30                      False                           0
6      MG02                                            92                             599                              46                        3.898836         2021-03-09T23:30:00+0800                         30                      False                           0
7      MG01                                            92                             592                              45                       20.000000         2021-03-09T23:30:00+0800                         30                      False                           0
8  apache02                                            82                             586                              31                       14.349109         2021-03-09T23:30:00+0800                         30                       True                           3
9  apache01                                            82                             548                              30                       25.808735         2021-03-09T23:30:00+0800                         30                       True                           3```
```